# ⚡ Speed team · Make the model faster without making it worse

**The challenge:** the fastest model wins, **as long as its CLER is no more than X points
above the baseline** (X is announced at the start of the day).

- **Speed** = median milliseconds for one prediction (1 second of EMG, 1 CPU thread).
- **CLER** = of the gestures the model detects, the share it names wrong (lower is better).

**You hand in** `demo_model.pt`, your best model. The judges score every entry on the
same machine with `speed_report.py`, so your numbers here are practice.

**Use quick mode while you try things.** `score()` below uses 5 one-minute windows spread
across each test user's session, 1–2 min per idea on Colab instead of 16. Quick numbers only
compare with other quick numbers, and they are noisier: if two ideas are within a few
tenths of a point, the official run decides.

**Heads-up:** INT8 is about 2.4× *faster* on Colab (Intel) but 8× *slower* on Apple
laptops. Always measure on the machine you're judged on.

**How to use this notebook**
1. **File → Save a copy in Drive.** You now have your own copy. Edit anything.
2. **Runtime → Run all**, or run cells one by one with ▶ / Shift+Enter.
3. The outputs you see before running are from the organizers' run, so you know what to expect.
   On Colab things take longer than shown (about 10× for the big runs).

*Data, model and the code underneath: Meta, [generic-neuromotor-interface](https://github.com/facebookresearch/generic-neuromotor-interface), **CC BY-NC 4.0** (non-commercial, keep the credit). Kaifosh, Reardon & CTRL-labs at Reality Labs, Nature (2025).*

In [ ]:
# @title ⚙️ Setup: run this first (1–2½ min). Double-click to see the code.
import base64, os, subprocess, sys, time, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
WORK = "/content/buildday" if IN_COLAB else os.path.abspath("buildday")
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
if WORK not in sys.path: sys.path.insert(0, WORK)
_t = time.time()
if "_T0" not in globals(): _T0 = _t   # start of the whole notebook run

FILES = {'requirements.txt': 'IyBXZWFyYWJsZSBBSSBCdWlsZCBEYXkgc3RhcnRlciByZXBvLgojIEluc3RhbGw6ICBwaXAgaW5zdGFsbCAtciByZXF1aXJlbWVudHMudHh0CiMgVGVzdGVkIHdpdGggUHl0aG9uIDMuMTIgKE1hY0Jvb2spIGFuZCBQeXRob24gMy4xMyAoQ29sYWIsIE5vdiAyMDI2KS4KIwojIE5vIHVwcGVyIGJvdW5kcyBvbiB0b3JjaC9udW1weSBvbiBwdXJwb3NlOiBDb2xhYiBhbHJlYWR5IGhhcyByZWNlbnQgdmVyc2lvbnMsCiMgYW5kIHBpbm5pbmcgb2xkIG9uZXMgbWFrZXMgcGlwIGRvd25sb2FkIH4yIEdCIGFuZCByZWluc3RhbGwgUHlUb3JjaC4KCiMgTWV0YSdzIGNvZGUsIHBpbm5lZCB0byB0aGUgY29tbWl0IHdlIHRlc3RlZCBhZ2FpbnN0IChDQyBCWS1OQyA0LjApCmdlbmVyaWNfbmV1cm9tb3Rvcl9pbnRlcmZhY2UgQCBnaXQraHR0cHM6Ly9naXRodWIuY29tL2ZhY2Vib29rcmVzZWFyY2gvZ2VuZXJpYy1uZXVyb21vdG9yLWludGVyZmFjZUBiNmJmMjUwZTJiZTVhNjdiMjM0ODgxMDQzMzUzNzNjZGI4N2ExNWM5CgojIFdoYXQgZW1nX2tpdCBhbmQgTWV0YSdzIGRhdGEvbWV0cmljIG1vZHVsZXMgaW1wb3J0CnRvcmNoPj0yLjQKbnVtcHk+PTEuMjYKbnVtYmE+PTAuNjAKaDVweT49My4xMQpwYW5kYXM+PTIuMgp0YWJsZXM+PTMuMTAgICAgICAgICAgIyBwYW5kYXMucmVhZF9oZGYgbmVlZHMgdGhpcyB0byByZWFkIHRoZSBnZXN0dXJlIGxhYmVscwpvbWVnYWNvbmY+PTIuMwp0b3JjaG1ldHJpY3M+PTAuMTEgICAgIyBpbXBvcnRlZCBieSBNZXRhJ3MgZGF0YSBtb2R1bGUgKGhhbmR3cml0aW5nIHV0aWxzKQpsZXZlbnNodGVpbj49MC4yNQp1bmlkZWNvZGU+PTEuMwpyZXF1ZXN0cwp0cWRtCgojIFBsb3RzLCBub3RlYm9va3MsIGRlbW8KbWF0cGxvdGxpYj49My44CnNlYWJvcm4+PTAuMTMKanVweXRlcmxhYj49NAppcHlrZXJuZWwKbmJmb3JtYXQKZ3JhZGlvPj01CgojIE5vdCBuZWVkZWQ6IHB5dG9yY2gtbGlnaHRuaW5nIC8gaHlkcmEtY29yZS4gVGhleSBhcmUgb25seSBmb3IgTWV0YSdzIHRyYWluaW5nCiMgbG9vcC4gVG8gcnVuIE1ldGEncyBvd24gdHJhaW5lci50ZXN0LCBzZWUgcmVxdWlyZW1lbnRzLW1ldGEudHh0Lgo=', 'emg_kit/__init__.py': '', 'emg_kit/core.py': '...', 'emg_kit/download.py': 'IiIiRG93bmxvYWQgTWV0YSdzIGRpc2NyZXRlLWdlc3R1cmVzIGRhdGEgYW5kIHByZXRyYWluZWQgY2hlY2twb2ludC4KClNhbWUgVVJMcyBhbmQgYmVoYXZpb3VyIGFzIE1ldGEncyBgZ2VuZXJpY19uZXVyb21vdG9yX2ludGVyZmFjZS9zY3JpcHRzL2Rvd25sb2FkXyoucHlgLgpXZSByZS1pbXBsZW1lbnQgdGhlIHR3byBlbnRyeSBwb2ludHMgaGVyZSBiZWNhdXNlIE1ldGEncyBgc2NyaXB0cy9gIGZvbGRlciBoYXMgbm8KYF9faW5pdF9fLnB5YCwgc28gaXQgaXMgbm90IGluY2x1ZGVkIHdoZW4gdGhlIHBhY2thZ2UgaXMgcGlwLWluc3RhbGxlZCBmcm9tIGdpdC4KClVzYWdlOgogICAgcHl0aG9uIC1tIGVtZ19raXQuZG93bmxvYWQgLS1zbWFsbC1zdWJzZXQgICAgICAgICAgIyAzIHVzZXJzLCB+MS4xIEdCCiAgICBweXRob24gLW0gZW1nX2tpdC5kb3dubG9hZCAgICAgICAgICAgICAgICAgICAgICAgICAgIyBmdWxsIGRhdGEsIH4zMyBHQiAob3JnYW5pemVycyBvbmx5KQogICAgcHl0aG9uIC1tIGVtZ19raXQuZG93bmxvYWQgLS1tb2RlbHMtb25seQoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgb3MKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIGdlbmVyaWNfbmV1cm9tb3Rvcl9pbnRlcmZhY2UuZG93bmxvYWRfdXRpbHMgaW1wb3J0ICgKICAgIGRvd25sb2FkX2ZpbGUsCiAgICBlbnN1cmVfZGlyLAogICAgZXh0cmFjdF90YXIsCikKCkJBU0VfVVJMID0gImh0dHBzOi8vZmItY3RybC1vc3MuczMuYW1hem9uYXdzLmNvbS9uZXVyb21vdG9yLWRhdGEiClRBU0sgPSAiZGlzY3JldGVfZ2VzdHVyZXMiCgoKZGVmIGRvd25sb2FkX2RhdGEoZW1nX2RhdGFfZGlyOiBzdHIgfCBQYXRoLCBkYXRhc2V0OiBzdHIgPSAic21hbGxfc3Vic2V0IiwKICAgICAgICAgICAgICAgICAgdGFzazogc3RyID0gVEFTSywgY2xlYW51cDogYm9vbCA9IFRydWUpIC0+IFBhdGg6CiAgICBkYXRhX2RpciA9IGVuc3VyZV9kaXIoUGF0aChlbWdfZGF0YV9kaXIpLmV4cGFuZHVzZXIoKSkKICAgIGZsYWcgPSBkYXRhX2RpciAvIGYiLmRhdGFfZG93bmxvYWRlZF97ZGF0YXNldH1fe3Rhc2t9IgogICAgaWYgZmxhZy5leGlzdHMoKToKICAgICAgICB3aGVuID0gZGF0ZXRpbWUuZnJvbXRpbWVzdGFtcChmbGFnLnN0YXQoKS5zdF9tdGltZSkKICAgICAgICBwcmludChmInt0YXNrfSB7ZGF0YXNldH0gYWxyZWFkeSBpbiB7ZGF0YV9kaXJ9IChkb3dubG9hZGVkIHt3aGVufSkuIikKICAgICAgICByZXR1cm4gZGF0YV9kaXIKCiAgICB0YXJfcGF0aCA9IGRhdGFfZGlyIC8gZiJ7dGFza31fe2RhdGFzZXR9LnRhciIKICAgIGRvd25sb2FkX2ZpbGUoZiJ7QkFTRV9VUkx9L2RhdGEve3Rhc2t9L3tkYXRhc2V0fS50YXIiLCB0YXJfcGF0aCwKICAgICAgICAgICAgICAgICAgZiJEb3dubG9hZGluZyB7ZGF0YXNldH0gZGF0YSBmb3Ige3Rhc2t9IikKICAgIGV4dHJhY3RfdGFyKHRhcl9wYXRoLCBkYXRhX2RpciwgIlVucGFja2luZyIpCiAgICBpZiBjbGVhbnVwOgogICAgICAgIHRhcl9wYXRoLnVubGluaygpCgogICAgY29ycHVzID0gZGF0YV9kaXIgLyBmInt0YXNrfV9jb3JwdXMuY3N2IgogICAgZG93bmxvYWRfZmlsZShmIntCQVNFX1VSTH0vZGF0YS97dGFza30ve3Rhc2t9X2NvcnB1cy5jc3YiLCBjb3JwdXMsCiAgICAgICAgICAgICAgICAgICJEb3dubG9hZGluZyBjb3JwdXMgQ1NWICh0cmFpbi92YWwvdGVzdCBzcGxpdCkiKQogICAgZmxhZy50b3VjaCgpCiAgICBwcmludChmIkRvbmU6IHtkYXRhX2Rpcn0iKQogICAgcmV0dXJuIGRhdGFfZGlyCgoKZGVmIGRvd25sb2FkX21vZGVscyhtb2RlbF9kaXI6IHN0ciB8IFBhdGgsIHRhc2s6IHN0ciA9IFRBU0spIC0+IFBhdGg6CiAgICB0YXNrX2RpciA9IGVuc3VyZV9kaXIoKFBhdGgobW9kZWxfZGlyKSAvIHRhc2spLmV4cGFuZHVzZXIoKSkKICAgIGlmICh0YXNrX2RpciAvICJtb2RlbF9jaGVja3BvaW50LmNrcHQiKS5leGlzdHMoKToKICAgICAgICBwcmludChmIkNoZWNrcG9pbnQgYWxyZWFkeSBpbiB7dGFza19kaXJ9LiIpCiAgICAgICAgcmV0dXJuIHRhc2tfZGlyCiAgICB0YXJfcGF0aCA9IHRhc2tfZGlyIC8gZiJ7dGFza30udGFyIgogICAgZG93bmxvYWRfZmlsZShmIntCQVNFX1VSTH0vY2hlY2twb2ludHMve3Rhc2t9L3t0YXNrfS50YXIiLCB0YXJfcGF0aCwKICAgICAgICAgICAgICAgICAgZiJEb3dubG9hZGluZyB7dGFza30gY2hlY2twb2ludCIpCiAgICBleHRyYWN0X3Rhcih0YXJfcGF0aCwgdGFza19kaXIsICJVbnBhY2tpbmciKQogICAgdGFyX3BhdGgudW5saW5rKCkKICAgIHByaW50KGYiRG9uZToge3Rhc2tfZGlyfSIpCiAgICByZXR1cm4gdGFza19kaXIKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyKQogICAgcC5hZGRfYXJndW1lbnQoIi0tZGF0YS1kaXIiLCBkZWZhdWx0PW9zLmVudmlyb24uZ2V0KCJFTUdfREFUQV9ESVIiLCAifi9lbWdfZGF0YSIpKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbW9kZWxzLWRpciIsIGRlZmF1bHQ9b3MuZW52aXJvbi5nZXQoIkVNR19NT0RFTFNfRElSIiwgIn4vZW1nX21vZGVscyIpKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc21hbGwtc3Vic2V0IiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgIGhlbHA9IjMgdXNlcnMgKH4xLjEgR0IpIGluc3RlYWQgb2YgdGhlIGZ1bGwgfjMzIEdCIGRhdGFzZXQiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbW9kZWxzLW9ubHkiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYSA9IHAucGFyc2VfYXJncygpCgogICAgZG93bmxvYWRfbW9kZWxzKGEubW9kZWxzX2RpcikKICAgIGlmIG5vdCBhLm1vZGVsc19vbmx5OgogICAgICAgIGRvd25sb2FkX2RhdGEoYS5kYXRhX2RpciwgInNtYWxsX3N1YnNldCIgaWYgYS5zbWFsbF9zdWJzZXQgZWxzZSAiZnVsbF9kYXRhIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==', 'speed_report.py': 'IiIiU2NvcmUgbW9kZWxzIGZvciB0aGUgU3BlZWQgY2hhbGxlbmdlOiBDTEVSLCBmaWxlIHNpemUsIGFuZCBDUFUgbGF0ZW5jeS4KCiAgICBweXRob24gc3BlZWRfcmVwb3J0LnB5IC0tbW9kZWwgYmFzZWxpbmUKICAgIHB5dGhvbiBzcGVlZF9yZXBvcnQucHkgLS1tb2RlbCBtb2RlbHMvaW50OF9keW5hbWljLnB0IC0tbW9kZWwgbW9kZWxzL3BydW5lZC5wdCAtLXggMS4wCiAgICBweXRob24gc3BlZWRfcmVwb3J0LnB5IC0tbW9kZWwgbW9kZWxzL21pbmUucHQgLS1xdWljayAgICAgIyBwcmFjdGljZTogfjZ4IGZhc3RlcgoKVGhlIGJhc2VsaW5lIGlzIGFsd2F5cyBzY29yZWQgaW4gdGhlIHNhbWUgcnVuIGFuZCB0aGUgc2FtZSBtb2RlLCBzbyBldmVyeQpjb21wYXJpc29uIGlzIGFwcGxlcyB0byBhcHBsZXMuIExhdGVuY3kgaXMgdGhlIG1lZGlhbiBtcyBmb3Igb25lIHByZWRpY3Rpb24gb3ZlcgoxIHMgb2YgMTYtY2hhbm5lbCBFTUcsIGJhdGNoIDEsIDEgQ1BVIHRocmVhZCAoc2VlIGVtZ19raXQuY29yZS5iZW5jaG1hcmtfbXMpLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB3YXJuaW5ncwoKaW1wb3J0IHRvcmNoCgp3YXJuaW5ncy5maWx0ZXJ3YXJuaW5ncygiaWdub3JlIikKCmZyb20gZW1nX2tpdC5jb3JlIGltcG9ydCAoUVVJQ0tfTUlOVVRFUywgYWxsX3Rlc3RfdXNlcnMsIGJlbmNobWFya19tcywgZXZhbHVhdGUsIGxvYWRfbW9kZWwsCiAgICAgICAgICAgICAgICAgICAgICAgICAgbW9kZWxfc2l6ZV9tYiwgdGVzdF9wYXJ0aXRpb25zKQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvcm1hdHRlcl9jbGFzcz1hcmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1tb2RlbCIsIGFjdGlvbj0iYXBwZW5kIiwgZGVmYXVsdD1bXSwKICAgICAgICAgICAgICAgICAgIGhlbHA9JyJiYXNlbGluZSIgb3IgYSBwYXRoIHRvIGEgLnB0IHNhdmVkIHdpdGggc2F2ZV93aG9sZV9tb2RlbC4gUmVwZWF0YWJsZS4nKQogICAgcC5hZGRfYXJndW1lbnQoIi0teCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgIGhlbHA9ImFsbG93ZWQgQ0xFUiBpbmNyZWFzZSBvdmVyIGJhc2VsaW5lLCBpbiBwb2ludHMgKDEuMCA9IDAuMDEgQ0xFUikiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tdXNlcnMiLCBuYXJncz0iKiIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0ibGltaXQgdG8gc29tZSB0ZXN0IHVzZXJzIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXF1aWNrIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgIGhlbHA9InByYWN0aWNlIG1vZGU6IDUgb25lLW1pbnV0ZSB3aW5kb3dzIHBlciB1c2VyICh+NnggZmFzdGVyLCBub3QgdGhlIG9mZmljaWFsIHNjb3JlKSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS10aHJlYWRzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MSwgaGVscD0iQ1BVIHRocmVhZHMgZm9yIHRoZSBsYXRlbmN5IHRlc3QiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0icmVzdWx0cy9zcGVlZF9yZXBvcnQuanNvbiIpCiAgICBhID0gcC5wYXJzZV9hcmdzKCkKCiAgICBuYW1lcyA9IFsiYmFzZWxpbmUiXSArIFttIGZvciBtIGluIGEubW9kZWwgaWYgbSAhPSAiYmFzZWxpbmUiXQogICAgaGF2ZSwgd2FudCA9IGxpc3QodGVzdF9wYXJ0aXRpb25zKCkpLCBhbGxfdGVzdF91c2VycygpCiAgICBwcmludChmIlRlc3QgdXNlcnM6IHtsZW4oaGF2ZSl9IG9mIHtsZW4od2FudCl9IGF2YWlsYWJsZSAoeycsICcuam9pbihoYXZlKX0pIikKICAgIGlmIGEucXVpY2s6CiAgICAgICAgcHJpbnQoZiJRVUlDSyBNT0RFOiB7UVVJQ0tfTUlOVVRFUzpnfSBtaW4gb2YgZWFjaCB1c2VyLiBQcmFjdGljZSBvbmx5LCAiCiAgICAgICAgICAgICAgIm5vdCB0aGUgb2ZmaWNpYWwgc2NvcmUuIikKICAgIHByaW50KGYiTGF0ZW5jeTogMSBzIHdpbmRvdywgYmF0Y2ggMSwge2EudGhyZWFkc30gdGhyZWFkKHMpXG4iKQoKICAgIHJvd3MgPSBbXQogICAgZm9yIG5hbWUgaW4gbmFtZXM6CiAgICAgICAgbW9kZWwgPSBsb2FkX21vZGVsKE5vbmUgaWYgbmFtZSA9PSAiYmFzZWxpbmUiIGVsc2UgbmFtZSkKICAgICAgICB0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGRmID0gZXZhbHVhdGUobW9kZWwsIHVzZXJzPWEudXNlcnMsIHZlcmJvc2U9RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgICBtaW51dGVzPVFVSUNLX01JTlVURVMgaWYgYS5xdWljayBlbHNlIE5vbmUpCiAgICAgICAgcm93cy5hcHBlbmQoewogICAgICAgICAgICAibW9kZWwiOiBuYW1lLAogICAgICAgICAgICAibWVhbl9jbGVyIjogZmxvYXQoZGYuY2xlci5tZWFuKCkpLAogICAgICAgICAgICAiYWNjdXJhY3kiOiBmbG9hdCgxIC0gZGYuY2xlci5tZWFuKCkpLAogICAgICAgICAgICAic2l6ZV9tYiI6IHJvdW5kKG1vZGVsX3NpemVfbWIobW9kZWwpLCAyKSwKICAgICAgICAgICAgIm1zX3Blcl9wcmVkaWN0aW9uIjogcm91bmQoYmVuY2htYXJrX21zKG1vZGVsLCB0aHJlYWRzPWEudGhyZWFkcyksIDMpLAogICAgICAgICAgICAidXNlcnMiOiBsaXN0KGRmLnVzZXIpLAogICAgICAgICAgICAicGVyX3VzZXJfY2xlciI6IGRpY3QoemlwKGRmLnVzZXIsIGRmLmNsZXIucm91bmQoNSkpKSwKICAgICAgICAgICAgImV2YWxfc2Vjb25kcyI6IHJvdW5kKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0LCAxKSwKICAgICAgICB9KQogICAgICAgIHByaW50KGYic2NvcmVkIHtuYW1lfSBpbiB7cm93c1stMV1bJ2V2YWxfc2Vjb25kcyddfXMiKQoKICAgIGJhc2UgPSByb3dzWzBdCiAgICBmb3IgciBpbiByb3dzOgogICAgICAgIHJbImNsZXJfcG9pbnRzX3ZzX2Jhc2VsaW5lIl0gPSByb3VuZCgoclsibWVhbl9jbGVyIl0gLSBiYXNlWyJtZWFuX2NsZXIiXSkgKiAxMDAsIDMpCiAgICAgICAgclsic3BlZWR1cCJdID0gcm91bmQoYmFzZVsibXNfcGVyX3ByZWRpY3Rpb24iXSAvIHJbIm1zX3Blcl9wcmVkaWN0aW9uIl0sIDIpCiAgICAgICAgclsiZWxpZ2libGUiXSA9IE5vbmUgaWYgYS54IGlzIE5vbmUgZWxzZSByWyJjbGVyX3BvaW50c192c19iYXNlbGluZSJdIDw9IGEueAoKICAgIHByaW50KGYiXG57J21vZGVsJzo8MzJ9eydDTEVSJzo+OH17J86UcHRzJzo+OH17J01CJzo+OH17J21zJzo+OX17J3ggZmFzdGVyJzo+MTB9eydvaz8nOj42fSIpCiAgICBmb3IgciBpbiByb3dzOgogICAgICAgIG9rID0gIiIgaWYgclsiZWxpZ2libGUiXSBpcyBOb25lIGVsc2UgKCJ5ZXMiIGlmIHJbImVsaWdpYmxlIl0gZWxzZSAiTk8iKQogICAgICAgIHByaW50KGYie1BhdGgoclsnbW9kZWwnXSkubmFtZTo8MzJ9e3JbJ21lYW5fY2xlciddOj44LjRmfXtyWydjbGVyX3BvaW50c192c19iYXNlbGluZSddOj4rOC4yZn0iCiAgICAgICAgICAgICAgZiJ7clsnc2l6ZV9tYiddOj44LjFmfXtyWydtc19wZXJfcHJlZGljdGlvbiddOj45LjJmfXtyWydzcGVlZHVwJ106PjEwLjJmfXtvazo+Nn0iKQoKICAgIGlmIGEueCBpcyBub3QgTm9uZToKICAgICAgICBvayA9IFtyIGZvciByIGluIHJvd3MgaWYgclsiZWxpZ2libGUiXV0gICAgICAgICAgIyBiYXNlbGluZSBpcyBhbHdheXMgZWxpZ2libGUKICAgICAgICB3ID0gbWluKG9rLCBrZXk9bGFtYmRhIHI6IHJbIm1zX3Blcl9wcmVkaWN0aW9uIl0pCiAgICAgICAgaWYgdyBpcyBiYXNlOgogICAgICAgICAgICBwcmludChmIlxuTm8gZW50cnkgd2l0aGluIHthLnh9IHB0cyBvZiBiYXNlbGluZSBpcyBmYXN0ZXIgdGhhbiB0aGUgYmFzZWxpbmUgIgogICAgICAgICAgICAgICAgICBmIih7YmFzZVsnbXNfcGVyX3ByZWRpY3Rpb24nXX0gbXMpLiIpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcHJpbnQoZiJcbldpbm5lciAod2l0aGluIHthLnh9IHB0cyBvZiBiYXNlbGluZSk6IHt3Wydtb2RlbCddfSAiCiAgICAgICAgICAgICAgICAgIGYiYXQge3dbJ21zX3Blcl9wcmVkaWN0aW9uJ119IG1zICh7d1snc3BlZWR1cCddfXggZmFzdGVyKSIpCgogICAgb3V0ID0gUGF0aChhLm91dCkKICAgIG91dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgbWV0YSA9IHsidGhyZWFkcyI6IGEudGhyZWFkcywgIngiOiBhLngsICJxdWljayI6IGEucXVpY2ssCiAgICAgICAgICAgICJ0b3JjaCI6IHRvcmNoLl9fdmVyc2lvbl9fLCAicXVhbnRfZW5naW5lIjogdG9yY2guYmFja2VuZHMucXVhbnRpemVkLmVuZ2luZX0KICAgIG91dC53cml0ZV90ZXh0KGpzb24uZHVtcHMoeyJtZXRhIjogbWV0YSwgInJlc3VsdHMiOiByb3dzfSwgaW5kZW50PTIpKQogICAgcHJpbnQoZiJcblNhdmVkIHtvdXR9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='}
for _f, _b in FILES.items():
    os.makedirs(os.path.dirname(_f) or ".", exist_ok=True)
    open(_f, "wb").write(base64.b64decode(_b))

def _run(args, what):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    if r.returncode:
        print((r.stdout + r.stderr)[-2500:])
        raise RuntimeError(f"{what} failed. Show this cell's output to a helper.")

if not os.path.exists(".installed"):
    _run(["-m", "pip", "install", "-q", "-r", "requirements.txt"], "Install")
    open(".installed", "w").close()
_run(["-m", "emg_kit.download", "--small-subset"], "Download")  # model + 3 test users, 1.1 GB

from emg_kit.core import *
import pandas as pd, numpy as np, matplotlib.pyplot as plt
USERS = ["user_000", "user_001", "user_002"]  # the 3 test users in the download
os.makedirs("handin", exist_ok=True)

print(f"✅ Ready in {time.time() - _t:.0f} s. Model and {len(USERS)} test users loaded.")

## 1 · Your scoreboard
`score(model, "name")` adds a row: quick CLER, Δ points vs. the baseline, size, speed.
The first row is the baseline.

In [ ]:
BOARD, MODELS = [], {}

def score(model, name):
    df = evaluate(model, users=USERS, minutes=QUICK_MINUTES, verbose=False)
    BOARD.append({"model": name, "quick CLER": round(df.cler.mean(), 4),
                  "MB": round(model_size_mb(model), 1), "ms": round(benchmark_ms(model), 1)})
    MODELS[name] = model
    b = pd.DataFrame(BOARD)
    b["Δ pts"] = ((b["quick CLER"] - b["quick CLER"][0]) * 100).round(2)
    b["× faster"] = (b.ms[0] / b.ms).round(2)
    return b

baseline = load_model()
score(baseline, "baseline (fp32)")

## 2 · First idea: INT8 quantization
Store the LSTM and output layer weights as 8-bit integers instead of 32-bit floats:
4× smaller, and on Intel CPUs, faster.

In [ ]:
score(quantize_int8(baseline), "int8 LSTM + Linear")

## 3 · Your ideas
Copy the pattern: build a model, then `score()` it. Some starting points:
- Quantize only part of the model (cells below).
- Look at `print(baseline)`: where does the time go? The conv layer, the 3 LSTM layers?
- Bigger ideas (smaller LSTM, pruning, distillation) need retraining on the full
  dataset with a GPU, which won't fit in a day. Pitch them in your slide instead.

In [ ]:
import torch
from torch import nn
lstm_only = torch.ao.quantization.quantize_dynamic(baseline, {nn.LSTM}, dtype=torch.qint8).eval()
score(lstm_only, "int8 LSTM only")

In [ ]:
linear_only = torch.ao.quantization.quantize_dynamic(baseline, {nn.Linear}, dtype=torch.qint8).eval()
score(linear_only, "int8 Linear only")

In [ ]:
# Your idea here

## 4 · Hand in your best model
Put the name of your best row in `BEST`. Then, if you have 30 minutes left, set
`RUN_OFFICIAL = True` in the next cell for the official check (full test data, the same
script the judges use).

In [ ]:
BEST = "int8 LSTM + Linear"        # <- the name from your scoreboard
save_whole_model(MODELS[BEST], "handin/demo_model.pt")
print(f"Saved {BEST} -> handin/demo_model.pt ({os.path.getsize('handin/demo_model.pt') / 1e6:.1f} MB)")

In [ ]:
RUN_OFFICIAL = False   # <- True for the official score (~30 min on Colab)
X = 2                  # <- the number announced in the morning

if RUN_OFFICIAL:
    !{sys.executable} speed_report.py --model handin/demo_model.pt --x {X}
else:
    print("Skipped. Set RUN_OFFICIAL = True to run the official score.")

In [ ]:
# Download your hand-in files (they also stay in the handin/ folder on the left).
print(f"⏱ Whole notebook so far: {time.time() - _T0:.0f} s")
import shutil
files_to_send = ['handin/demo_model.pt']
missing = [f for f in files_to_send if not os.path.exists(f)]
if missing:
    print("Not made yet:", missing)
elif IN_COLAB:
    from google.colab import files
    for f in files_to_send:
        files.download(f)
else:
    print("Saved:", files_to_send)